# K-Nearest Neighbors (KNN) Classifier — Scratch (Tanpa scikit-learn)

Notebook ini implementasi **KNN** dari nol (tanpa `sklearn`) untuk dataset:
- `dataset_dt_2kelas_3var_100.csv`

Fitur (X):
- `luas_tanah`
- `luas_bangunan`
- `jarak_kota`

Target (Y):
- `kelas` (Murah / Mahal)

> **Catatan penting:** KNN sensitif terhadap skala fitur, jadi kita pakai **normalisasi Z-score** (mean & std dari data train).


## 1) Import library

In [8]:
import pandas as pd
import numpy as np

## 2) Load data (pd.read_csv)

In [9]:
df = pd.read_csv('/PyAi/datasetrumah.csv')

df.head()

,id,luas_tanah,luas_bangunan,jarak_kota,kelas
0,H0204,174,75,6.0,Mahal
1,H0267,217,118,2.8,Mahal
2,H0153,127,87,3.1,Mahal
3,H0010,154,133,16.3,Murah
4,H0234,193,119,5.5,Mahal


## 3) Pilih fitur (X) dan target (Y)

In [10]:
FEATURES = ["luas_tanah", "luas_bangunan", "jarak_kota"]
TARGET = "kelas"

X = df[FEATURES].to_numpy(dtype=float)
y = df[TARGET].to_numpy()

print("Jumlah data:", len(df))
print("Fitur:", FEATURES)
print("Target:", TARGET)
print("Label unik:", np.unique(y))

Jumlah data: 300
Fitur: ['luas_tanah', 'luas_bangunan', 'jarak_kota']
Target: kelas
Label unik: ['Mahal' 'Murah']


## 4) Split data (train/test)

In [11]:
def train_test_split(X, y, test_size=0.25, seed=123):
    # Membuat random number generator dengan seed tertentu
    # Seed digunakan agar hasil pengacakan selalu sama (reproducible)
    rng = np.random.default_rng(seed)
    
    # Membuat array indeks dari 0 sampai panjang data X - 1
    idx = np.arange(len(X))
    
    # Mengacak urutan indeks
    rng.shuffle(idx)
    
    # Menentukan titik pemisah antara data train dan test
    # (1 - test_size) artinya proporsi data latih
    split = int(len(X) * (1 - test_size))
    
    # Indeks untuk data training (bagian awal)
    train_idx = idx[:split]
    
    # Indeks untuk data testing (bagian akhir)
    test_idx = idx[split:]
    
    # Mengembalikan data X dan y yang sudah dibagi
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


# Memanggil fungsi train_test_split
# Data dibagi menjadi 75% training dan 25% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, seed=123)

# Menampilkan jumlah data training dan testing
print("Train:", len(X_train), "Test:", len(X_test))

Train: 225 Test: 75


## 5) Normalisasi Z-score (fit di train, transform train & test)

In [12]:
def zscore_fit(X_train):
    # Menghitung rata-rata (mean) tiap kolom/fitur pada data training
    # axis=0 artinya dihitung per kolom (per fitur)
    mu = X_train.mean(axis=0)
    
    # Menghitung standar deviasi tiap kolom/fitur pada data training
    sigma = X_train.std(axis=0)
    
    # Jika ada standar deviasi bernilai 0, ubah menjadi 1
    # Tujuannya agar tidak terjadi pembagian dengan nol saat normalisasi
    sigma = np.where(sigma == 0, 1.0, sigma)
    
    # Mengembalikan nilai mean dan standar deviasi
    return mu, sigma


def zscore_transform(X, mu, sigma):
    # Melakukan standardisasi Z-score:
    # (nilai - mean) / standar deviasi
    return (X - mu) / sigma


# Menghitung mean dan standar deviasi dari data training saja
mu, sigma = zscore_fit(X_train)

# Menstandarkan data training menggunakan mu dan sigma
X_train_s = zscore_transform(X_train, mu, sigma)

# Menstandarkan data testing menggunakan mu dan sigma yang sama
# (PENTING agar tidak terjadi data leakage)
X_test_s = zscore_transform(X_test, mu, sigma)

# Menampilkan mean data training
print("Mean (train):", mu)

# Menampilkan standar deviasi data training
print("Std  (train):", sigma)

Mean (train): [130.96444444  88.31111111   8.93244444]
Std  (train): [41.17084813 27.74712976  4.09180925]


## 6) KNN dari nol (Euclidean distance + majority vote)

In [13]:
def knn_predict_one(x, X_train, y_train, k=5):
    # Menghitung jarak Euclidean antara 1 data uji (x)
    # dengan seluruh data training
    # (X_train - x) → selisih tiap fitur
    # **2 → kuadratkan
    # sum(axis=1) → jumlahkan per baris (per data training)
    # sqrt → akar kuadrat (rumus jarak Euclidean)
    dists = np.sqrt(np.sum((X_train - x) ** 2, axis=1))
    
    # Mengurutkan jarak dari yang paling kecil
    # Lalu mengambil indeks k tetangga terdekat
    nn_idx = np.argsort(dists)[:k]
    
    # Mengambil label dari k tetangga terdekat
    nn_labels = y_train[nn_idx]

    # Voting mayoritas:
    # np.unique → nilai label unik
    # return_counts=True → hitung jumlah tiap label
    values, counts = np.unique(nn_labels, return_counts=True)
    
    # Mengambil label dengan jumlah kemunculan terbanyak
    return values[np.argmax(counts)]


def knn_predict(X_test, X_train, y_train, k=5):
    # Melakukan prediksi untuk setiap data uji
    # List comprehension → memanggil knn_predict_one untuk tiap x
    # np.array → hasil dikonversi menjadi array numpy
    return np.array([
        knn_predict_one(x, X_train, y_train, k=k)
        for x in X_test
    ])

## 7) Evaluasi (Confusion Matrix, Accuracy, Precision, Recall, F1) — tanpa sklearn

In [14]:
def accuracy(y_true, y_pred):
    # Membandingkan label asli (y_true) dan prediksi (y_pred)
    # y_true == y_pred menghasilkan array True/False
    # np.mean menghitung proporsi True → akurasi
    return np.mean(y_true == y_pred)

def confusion_matrix_df(y_true, y_pred, labels=None):
    # Jika label tidak diberikan,
    # ambil semua label unik dari y_true dan y_pred
    if labels is None:
        labels = np.unique(np.concatenate([y_true, y_pred]))
    
    # Ubah label menjadi list
    labels = list(labels)
    
    # Membuat mapping label → indeks matriks
    label_to_idx = {lab: i for i, lab in enumerate(labels)}
    
    # Membuat confusion matrix kosong (ukuran n_label x n_label)
    cm = np.zeros((len(labels), len(labels)), dtype=int)

    # Mengisi confusion matrix
    # Baris = label sebenarnya (True)
    # Kolom = label prediksi (Pred)
    for yt, yp in zip(y_true, y_pred):
        cm[label_to_idx[yt], label_to_idx[yp]] += 1

    # Mengubah ke DataFrame agar lebih rapi dibaca
    return pd.DataFrame(
        cm,
        index=[f"True_{l}" for l in labels],
        columns=[f"Pred_{l}" for l in labels]
    )

def classification_report_df(y_true, y_pred, labels=None):
    # Jika label tidak diberikan, ambil label unik
    if labels is None:
        labels = np.unique(np.concatenate([y_true, y_pred]))
    labels = list(labels)

    # Menyimpan hasil per kelas
    rows = []
    
    # Total data
    support_total = 0
    
    # Variabel untuk macro average
    macro_p = macro_r = macro_f1 = 0.0
    
    # Variabel untuk weighted average
    weighted_p = weighted_r = weighted_f1 = 0.0

     for lab in labels:
        # True Positive
        tp = np.sum((y_true == lab) & (y_pred == lab))
        
        # False Positive
        fp = np.sum((y_true != lab) & (y_pred == lab))
        
        # False Negative
        fn = np.sum((y_true == lab) & (y_pred != lab))
        
        # Jumlah data sebenarnya untuk kelas ini
        support = np.sum(y_true == lab)
        support_total += support

              # Precision = TP / (TP + FP)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        
        # Recall = TP / (TP + FN)
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        
        # F1-score = harmonic mean precision & recall
        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall) > 0 else 0.0
        )

               # Simpan hasil per kelas
        rows.append([lab, precision, recall, f1, support])

        # Akumulasi untuk macro average
        macro_p += precision
        macro_r += recall
        macro_f1 += f1

        # Akumulasi untuk weighted average
        weighted_p += precision * support
        weighted_r += recall * support
        weighted_f1 += f1 * support
        weighted_f1 += f1 * support

    # Jumlah kelas
    n_classes = len(labels)
    
    # Macro average = rata-rata tanpa bobot
    macro_p /= n_classes
    macro_r /= n_classes
    macro_f1 /= n_classes

    # Weighted average = dibobot oleh jumlah data per kelas
    if support_total > 0:
        weighted_p /= support_total
        weighted_r /= support_total
        weighted_f1 /= support_total

       # DataFrame hasil per kelas
    report = pd.DataFrame(
        rows,
        columns=["class", "precision", "recall", "f1_score", "support"]
    )

    # Baris ringkasan
    summary = pd.DataFrame([
        ["macro_avg", macro_p, macro_r, macro_f1, support_total],
        ["weighted_avg", weighted_p, weighted_r, weighted_f1, support_total],
    ], columns=report.columns)

    # Gabungkan laporan kelas + ringkasan
    return pd.concat([report, summary], ignore_index=True)

## 8) Training + hasil evaluasi

In [15]:
k = 5  # kamu bisa coba 3, 5, 7, dst.

y_pred = knn_predict(X_test_s, X_train_s, y_train, k=k)

labels = np.unique(np.concatenate([y_test, y_pred]))
cm = confusion_matrix_df(y_test, y_pred, labels=labels)
report = classification_report_df(y_test, y_pred, labels=labels)

print(f"KNN (k={k})")
print("Accuracy:", accuracy(y_test, y_pred))

print("\nConfusion Matrix:")
display(cm)

print("\nPrecision / Recall / F1:")
display(report)

KNN (k=5)
Accuracy: 1.0

Confusion Matrix:


,Pred_Mahal,Pred_Murah
True_Mahal,40,0
True_Murah,0,35



Precision / Recall / F1:


,class,precision,recall,f1_score,support
0,Mahal,1.0,1.0,1.0,40
1,Murah,1.0,1.0,1.0,35
2,macro_avg,1.0,1.0,1.0,75
3,weighted_avg,1.0,1.0,1.0,75


## 9) Prediksi data baru (contoh)

In [16]:
contoh = np.array([
    [90, 70, 12.0],
    [180, 130, 3.0],
], dtype=float)

contoh_s = zscore_transform(contoh, mu, sigma)
pred_contoh = knn_predict(contoh_s, X_train_s, y_train, k=k)

pd.DataFrame(contoh, columns=FEATURES).assign(prediksi=pred_contoh)

,luas_tanah,luas_bangunan,jarak_kota,prediksi
0,90.0,70.0,12.0,Murah
1,180.0,130.0,3.0,Mahal


## Perbandingan hasil untuk berbagai nilai **k**

Sel ini membuat tabel evaluasi (akurasi) untuk beberapa nilai **k** agar kamu bisa melihat kecenderungan performa dan memilih **k** yang “ideal”.
> Catatan: ini memakai data train/test split yang sudah dibuat di sel sebelumnya (variabel `X_train`, `X_test`, `y_train`, `y_test`).


In [17]:
import numpy as np
import pandas as pd

def knn_predict_one(X_train, y_train, x_query, k):
    """Prediksi KNN untuk 1 data (tanpa sklearn).
    Tie-break: kalau seri, ambil label tetangga paling dekat.
    """
    dists = np.sqrt(((X_train - x_query) ** 2).sum(axis=1))
    idx = np.argsort(dists)[:k]
    neigh_labels = y_train[idx]

    # majority vote
    values, counts = np.unique(neigh_labels, return_counts=True)
    max_count = counts.max()
    winners = values[counts == max_count]

    if len(winners) == 1:
        pred = winners[0]
    else:
        pred = neigh_labels[0]  # tie-break: nearest neighbor

    return pred, idx, dists[idx]

# ====== pilih 1 record query ======
q_idx = 0
x_query = X_test[q_idx]
label_asli = y_test[q_idx]   # << ini yang kamu butuhin

print("Label asli query:", label_asli)

# rentang k yang mau dibandingin
max_k = min(15, len(X_train))
k_values = list(range(1, max_k + 1))

rows = []
for k in k_values:
    pred, idx, dist_k = knn_predict_one(X_train, y_train, x_query, k)

    rows.append([
        k,
        label_asli,
        pred,
        "BENAR" if pred == label_asli else "SALAH",
        list(y_train[idx]),
        [float(f"{d:.4f}") for d in dist_k]
    ])

df_k = pd.DataFrame(
    rows,
    columns=["k", "label_asli", "prediksi", "status", "label_tetangga", "jarak_tetangga"]
)

display(df_k)

Label asli query: Murah


,k,label_asli,prediksi,status,label_tetangga,jarak_tetangga
0,1,Murah,Mahal,SALAH,[Mahal],[6.7]
1,2,Murah,Mahal,SALAH,"[Mahal, Murah]","[6.7, 7.0434]"
2,3,Murah,Mahal,SALAH,"[Mahal, Murah, Mahal]","[6.7, 7.0434, 9.4868]"
3,4,Murah,Mahal,SALAH,"[Mahal, Murah, Mahal, Mahal]","[6.7, 7.0434, 9.4868, 9.5399]"
4,5,Murah,Mahal,SALAH,"[Mahal, Murah, Mahal, Mahal, Mahal]","[6.7, 7.0434, 9.4868, 9.5399, 9.6026]"
5,6,Murah,Mahal,SALAH,"[Mahal, Murah, Mahal, Mahal, Mahal, Mahal]","[6.7, 7.0434, 9.4868, 9.5399, 9.6026, 9.8914]"
6,7,Murah,Mahal,SALAH,"[Mahal, Murah, Mahal, Mahal, Mahal, Mahal, Mahal]","[6.7, 7.0434, 9.4868, 9.5399, 9.6026, 9.8914, ..."
7,8,Murah,Mahal,SALAH,"[Mahal, Murah, Mahal, Mahal, Mahal, Mahal, Mah...","[6.7, 7.0434, 9.4868, 9.5399, 9.6026, 9.8914, ..."
8,9,Murah,Mahal,SALAH,"[Mahal, Murah, Mahal, Mahal, Mahal, Mahal, Mah...","[6.7, 7.0434, 9.4868, 9.5399, 9.6026, 9.8914, ..."
9,10,Murah,Mahal,SALAH,"[Mahal, Murah, Mahal, Mahal, Mahal, Mahal, Mah...","[6.7, 7.0434, 9.4868, 9.5399, 9.6026, 9.8914, ..."
